# Multimodal Systems - Project 10
You have been asked to develop an interactive system for an art installation: when a visitor enters the room, two “walls” appear on the right and left sides of the visitor’s bounding rectangle, attempting to crush their body. The visitor can push away the walls by enlarging the bounding rectangle, thus seeing the background of the room. If the visitor’s energy is lower than a given threshold, the visitor disappears, and the two walls join at the center of the image. You can build the wall by drawing two rectangles: one from the left edge of the image to the left edge of the bounding rectangle, and one from the right edge of the bounding rectangle to the right edge of the image.
1. Identify the sensor devices you can use to develop the system.
2. Design the system and identify possible critical aspects in its implementation.
3. Build an initial prototype of this system in Python. 

In [9]:
import cv2
import sys
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
def get_bounding_rectangle(rgb_image, detection_result):
    height, width, _ = rgb_image.shape

    xs = [lm.x for lm in detection_result.pose_landmarks[0]]
    ys = [lm.y for lm in detection_result.pose_landmarks[0]]

    x_min = int(min(xs) * width)
    x_max = int(max(xs) * width)
    y_min = int(min(ys) * height)
    y_max = int(max(ys) * height)

    rectangle = np.array([[x_min, y_min], [x_min, y_max], [x_max, y_max], [x_max, y_min]], dtype=np.int32)

    return rectangle

# DEBUG
def draw_bounding_rectangle(mask, rectangle, color=(0, 255, 0), thickness=2, fill=False):
    if rectangle is None or rectangle.shape != (4, 2):
        return mask

    # Ensure integer coordinates
    pts = np.array([[int(rectangle[i, 0]), int(rectangle[i, 1])] for i in range(4)], dtype=np.int32).reshape((-1, 1, 2))

    # Make a copy to avoid modifying original mask
    mask_copy = mask.copy()

    if fill:
        cv2.fillPoly(mask_copy, [pts], color)
    else:
        cv2.polylines(mask_copy, [pts], isClosed=True, color=color, thickness=thickness)

    return mask_copy

# DEBUG
def draw_landmarks_on_image(rgb_image, detection_result, draw_connections=True):
    annotated_image = rgb_image.copy()

    # Check if any poses were detected
    if detection_result.pose_landmarks:
        for pose_landmarks in detection_result.pose_landmarks:
            # Convert normalized landmarks to pixel coordinates
            h, w, _ = annotated_image.shape
            landmark_points = []
            for lm in pose_landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                landmark_points.append((x, y))
                cv2.circle(annotated_image, (x, y), 3, (0, 255, 0), -1)  # small green dot

            # Draw connections if requested
            if draw_connections:
                # Define the typical pose connections (subset)
                POSE_CONNECTIONS = [
                    (0, 1), (1, 2), (2, 3), (3, 7),  # Head/neck
                    (0, 4), (4, 5), (5, 6), (6, 8),  # Other side
                    (9, 10), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), # torso/limbs
                ]
                for start_idx, end_idx in POSE_CONNECTIONS:
                    if start_idx < len(landmark_points) and end_idx < len(landmark_points):
                        cv2.line(annotated_image, landmark_points[start_idx], landmark_points[end_idx], (0, 255, 255), 2)

    return annotated_image

In [11]:
# Constants
video_path = "videos\micro-dance.avi"
live_input = False

In [ ]:
# Creating a PoseLandmarker object
base_options = python.BaseOptions(model_asset_path='models\\pose_landmarker_lite.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)

# Selecting the video camera as input source
cap = cv2.VideoCapture(0)
print("Processing webcam input.")

# Checking for possible errors
if not cap.isOpened():
    print("Error in opening the video stream.")
    sys.exit()

# image resize 
cap.set(3, 640)
cap.set(4, 480)

while True:
    
    # Getting current frame
    success, current_frame = cap.read()
    current_frame = cv2.flip(current_frame, 1) # mirror

    # For each frame, detect landmarks 
    if success:
        #we change the image format for compatibility (some precious time is wasted)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=current_frame)
        detection_result = detector.detect(mp_image)

        if detection_result.pose_landmarks:
            
            if len(detection_result.pose_landmarks) > 1:
                # Text parameters
                text = "WARNING: Someone is disturbing the game!"
                position = (50, 50)             
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.8
                color = (0, 0, 255)             
                thickness = 2

                cv2.putText(current_frame, text, position, font, font_scale, color, thickness, cv2.LINE_AA)

            else:
                annotated_image = draw_landmarks_on_image(current_frame, detection_result) # DEBUG

                bounding_rect = get_bounding_rectangle(current_frame, detection_result)
                annotated_image = draw_bounding_rectangle(annotated_image, bounding_rect) # DEBUG
        
        else:
            annotated_image = current_frame

        cv2.imshow("Mediapipe", annotated_image)

        if cv2.waitKey(1) & 0xFF==ord('q'): # quit when 'q' is pressed
            cap.release()
            break
    else:
        break

# Closing video capture device
cv2.destroyAllWindows() 
cv2.waitKey(1)

Processing webcam input.


-1